# Day 5: Object-oriented Python and exceptions (Solutions)

**Python for Everybody:** Course 4 *Using Databases with Python*, chapter 14 (Object-oriented programming), plus the exception material in chapters 3 and 7.

**Why it matters for the JD:** 'Own components from design through production support' and 'write clean, modular, reusable code' means classes, clear error handling and context managers. OOP questions appear in nearly every Python interview.

**Time:** about 90 minutes · 7 exercises

These are reference solutions. Try the exercises notebook first, then compare. Every cell here passes its check.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

TOTAL = 7

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))


Run the next cell for the imports used in this day.

In [ ]:
import math
import time

## Exercise 1: Predict the output: try / except / else / finally  `warm-up`

Set `answer` to the list `f()` returns.

```python
def f():
    log = []
    try:
        log.append("try")
        1 / 0
    except ZeroDivisionError:
        log.append("except")
    else:
        log.append("else")
    finally:
        log.append("finally")
    return log
```

In [ ]:
answer = ["try", "except", "finally"]

In [ ]:
def _test():
    def f():
        log = []
        try:
            log.append("try")
            1 / 0
        except ZeroDivisionError:
            log.append("except")
        else:
            log.append("else")
        finally:
            log.append("finally")
        return log

    assert answer == f(), "Trace it step by step. Did the try block raise?"

check('1. Predict the output: try / except / else / finally', _test)

## Exercise 2: Task class: attributes, methods, __str__  `easy`

Write a `Task` class (the start of your Task Manager project):
- `Task(task_id, title, done=False)` stores `id`, `title`, `done`
- `complete()` sets `done` to `True`
- `str(task)` gives `"[ ] 1: Buy milk"` (or `"[x] 1: Buy milk"` when done)

In [ ]:
class Task:
    def __init__(self, task_id, title, done=False):
        self.id = task_id
        self.title = title
        self.done = done

    def complete(self):
        self.done = True

    def __str__(self):
        mark = "x" if self.done else " "
        return f"[{mark}] {self.id}: {self.title}"

In [ ]:
def _test():
    t = Task(1, "Buy milk")
    assert (t.id, t.title, t.done) == (1, "Buy milk", False), "attributes should be id, title, done (default False)"
    assert str(t) == "[ ] 1: Buy milk"
    t.complete()
    assert t.done is True
    assert str(t) == "[x] 1: Buy milk"

check('2. Task class: attributes, methods, __str__', _test)

## Exercise 3: Money: __repr__, __str__, __eq__, __add__  `medium`

Write `Money(paise, currency="INR")`, storing whole paise as an integer:
- `repr(Money(1050))` -> `"Money(1050, 'INR')"`
- `str(Money(1050))` -> `"INR 10.50"`
- `==` compares paise **and** currency; comparing with a non-Money is just `False`
- `+` adds two Money values, but raises `ValueError` if the currencies differ

In [ ]:
class Money:
    def __init__(self, paise, currency="INR"):
        self.paise = paise
        self.currency = currency

    def __repr__(self):
        return f"Money({self.paise}, {self.currency!r})"

    def __str__(self):
        return f"{self.currency} {self.paise / 100:.2f}"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.paise, self.currency) == (other.paise, other.currency)

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("currency mismatch")
        return Money(self.paise + other.paise, self.currency)

In [ ]:
def _test():
    m = Money(1050)
    assert repr(m) == "Money(1050, 'INR')", "repr should look like Money(1050, 'INR')"
    assert str(m) == "INR 10.50"
    assert Money(100) == Money(100) and Money(100) != Money(101)
    assert Money(100, "USD") != Money(100), "different currency means not equal"
    assert (Money(100) == 5) is False
    assert Money(100) + Money(250) == Money(350)
    assert raises(ValueError, lambda: Money(100, "USD") + Money(100, "INR")), "adding different currencies must raise ValueError"

check('3. Money: __repr__, __str__, __eq__, __add__', _test)

## Exercise 4: Shapes: inheritance and polymorphism  `medium`

`Shape` is given. Write `Rectangle(width, height)` and `Circle(radius)` as subclasses that override `area()`. Then write `total_area(shapes)` returning the sum of the areas rounded to 2 decimals. `total_area` must work on **any** shape without checking its type (that's polymorphism).

In [ ]:
class Shape:
    def area(self):
        raise NotImplementedError


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


def total_area(shapes):
    return round(sum(shape.area() for shape in shapes), 2)

In [ ]:
def _test():
    assert issubclass(Rectangle, Shape) and issubclass(Circle, Shape), "both should inherit from Shape"
    assert Rectangle(2, 3).area() == 6
    assert abs(Circle(1).area() - math.pi) < 1e-9
    assert total_area([Rectangle(2, 3), Circle(1)]) == 9.14
    assert total_area([]) == 0
    assert raises(NotImplementedError, Shape().area)

check('4. Shapes: inheritance and polymorphism', _test)

## Exercise 5: Account: @property and a custom exception  `medium`

Write a custom `InsufficientFunds(Exception)` and an `Account` class:
- `Account(balance=0)` keeps the balance in `_balance`
- `balance` is a **read-only property** (setting it raises `AttributeError`)
- `deposit(amount)`: amount must be positive, else `ValueError`
- `withdraw(amount)`: amount must be positive (`ValueError`); if it's more than the balance raise `InsufficientFunds`

In [ ]:
class InsufficientFunds(Exception):
    pass


class Account:
    def __init__(self, balance=0):
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        if amount > self._balance:
            raise InsufficientFunds(f"balance {self._balance} is less than {amount}")
        self._balance -= amount

In [ ]:
def _test():
    acct = Account(100)
    assert acct.balance == 100
    acct.deposit(50)
    acct.withdraw(30)
    assert acct.balance == 120
    assert raises(AttributeError, setattr, acct, "balance", 5), "balance must be read-only"
    assert raises(InsufficientFunds, acct.withdraw, 500)
    assert acct.balance == 120, "a failed withdrawal must not change the balance"
    assert raises(ValueError, acct.deposit, 0)
    assert raises(ValueError, acct.withdraw, -5)
    assert issubclass(InsufficientFunds, Exception)

check('5. Account: @property and a custom exception', _test)

## Exercise 6: Timer: write a context manager  `medium`

Write a class `Timer` usable as `with Timer() as t:`. After the block, `t.elapsed` holds the seconds spent (a float). If the block raises, the exception must **still propagate** (don't swallow it), and `elapsed` must still be set.

In [ ]:
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self.start
        return False

In [ ]:
def _test():
    with Timer() as t:
        sum(range(1000))
    assert isinstance(t.elapsed, float) and t.elapsed >= 0, "elapsed should be a non-negative float"
    t2 = Timer()
    try:
        with t2:
            raise ValueError("boom")
    except ValueError:
        pass
    else:
        raise AssertionError("the ValueError must propagate out of the with block")
    assert hasattr(t2, "elapsed"), "elapsed should be set even when the block raised"

check('6. Timer: write a context manager', _test)

## Exercise 7: Interview: TaskManager class  `interview`

Using your `Task` class from exercise 2, write `TaskManager`:
- `add(title)` creates a `Task` with ids 1, 2, 3, ... and returns it
- `complete(task_id)` marks that task done; an unknown id raises `KeyError`
- `pending()` returns the not-done tasks as a list, in the order they were added
- `len(manager)` is the number of tasks

This is the core of your project, and a very common 'design a small class' interview task.

In [ ]:
class TaskManager:
    def __init__(self):
        self._tasks = {}
        self._next_id = 1

    def add(self, title):
        task = Task(self._next_id, title)
        self._tasks[task.id] = task
        self._next_id += 1
        return task

    def complete(self, task_id):
        self._tasks[task_id].complete()

    def pending(self):
        return [t for t in self._tasks.values() if not t.done]

    def __len__(self):
        return len(self._tasks)

In [ ]:
def _test():
    tm = TaskManager()
    a = tm.add("write code")
    b = tm.add("test code")
    assert (a.id, b.id) == (1, 2), "ids should count 1, 2, 3 ..."
    assert len(tm) == 2
    tm.complete(1)
    assert [t.title for t in tm.pending()] == ["test code"]
    assert raises(KeyError, tm.complete, 99), "an unknown id should raise KeyError"

check('7. Interview: TaskManager class', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, then compare.

**What are the four pillars of OOP, in Python terms?**

Encapsulation (bundle data and behaviour, `_private` by convention, properties), abstraction (hide details behind a simple interface, `abc`), inheritance (`class B(A)`, `super()`), polymorphism (the same call works on different types: duck typing).

**`__str__` vs `__repr__`?**

`__repr__` is for developers: unambiguous, ideally valid code (`Money(1050, 'INR')`). `__str__` is for users: readable (`INR 10.50`). `print` uses `__str__`; the REPL uses `__repr__`; if `__str__` is missing Python falls back to `__repr__`.

**`@classmethod` vs `@staticmethod` vs a normal method?**

A normal method gets the instance (`self`). A classmethod gets the class (`cls`), often used for alternative constructors. A staticmethod gets neither: it is just a function namespaced in the class.

**What does `finally` guarantee?**

It runs whether or not an exception happened, even on `return` inside `try`. Use it for cleanup, though `with` is usually cleaner.

**Why create custom exceptions?**

They let callers catch exactly your failure (`except InsufficientFunds`) without catching unrelated errors, and they give errors a clear domain name.

